In [1]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
from torch.distributions import Categorical
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)

In [2]:
def preprocess_frame(frame):
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    frame = cv2.resize(frame, (84, 84))
    frame = frame / 255.0
    return frame

# Stack 4 frames together to create temporal awareness for the agent
def stack_frames(stacked_frames, new_frame, is_new_episode):
    if is_new_episode:
        stacked_frames = deque([np.zeros((84, 84), dtype=np.float32) for _ in range(4)], maxlen=4)
        for _ in range(4):
            stacked_frames.append(new_frame)
    else:
        stacked_frames.append(new_frame)

    stacked_state = np.stack(stacked_frames, axis=0)
    return stacked_state, stacked_frames

In [3]:
c = deque([np.zeros((84, 84), dtype=np.float32) for i in range(10)])
c[0].shape
stacked_state = np.stack(c, axis=0)
stacked_state.shape

(10, 84, 84)

In [ ]:
class A2CNetwork(nn.Module):
    def __init__(self, input_channels, action_space):
        super(A2CNetwork, self).__init__()

        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)

        self.fc1 = nn.Linear(64 * 7 * 7, 512)

        # Seperate output heads for actor and critic
        self.actor = nn.Linear(512, action_space)
        self.critic = nn.Linear(512, 1)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        # I didn't understand the shape of entery to nn.Linear!
        # x = x.view(x.size(0), -1)
        x = x.flatten()
        x = torch.relu(self.fc1(x))

        policy = self.actor(x)
        value = self.critic(x)
        return policy, value


In [5]:
class A2CAgent:

    def __init__(self, env, network, optimizer, gamma=0.99):
        self.env = env
        self.network = network
        self.optimizer = optimizer
        self.gamma = gamma
        self.stacked_frame = deque(maxlen=4)

    def preprocess_state(self, state, is_new_episode):
        processed_frame = preprocess_frame(state)
        stacked_state, self.stacked_frame = stack_frames(self.stacked_frame, processed_frame, is_new_episode)
        return stacked_state

    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0)
        policy, value = self.network(state)
        probs = torch.softmax(policy, dim=-1)
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action), value

    def compute_returns(self, rewards):
        returns = []
        R = 0
        for r in reversed(rewards):
            R = r + self.gamma * R
            returns.insert(0, R)
        return returns

    def train(self, num_episodes=1000):
        for episode in range(num_episodes):
            state, _ = self.env.reset()

            # print(f'state getting out of env: {state.shape}')

            state = self.preprocess_state(state, is_new_episode=True)

            # print(f'state getting out of preprocess: {state.shape}')

            log_probs = []
            values = []
            rewards = []

            done = False
            while not done:
                action, log_prob, value = self.select_action(state)
                next_state, reward, done, _, _ = self.env.step(action)
                next_state = self.preprocess_state(next_state, is_new_episode=False)

                log_probs.append(log_prob)
                values.append(value)
                rewards.append(reward)

                state = next_state

            returns = self.compute_returns(rewards)
            policy_loss = []
            value_loss = []
            for log_prob, value, R in zip(log_probs, values, returns):
                advantage = R - value.item()
                policy_loss.append(-log_prob * advantage)
                value_loss.append(nn.functional.mse_loss(value, torch.tensor([R])))

            self.optimizer.zero_grad()
            # !!!!!!!!!!!!!!!!!!!!!!! why use stack?
            loss = torch.stack(policy_loss).sum() + torch.stack(value_loss).sum()
            loss.backward()
            self.optimizer.step()

            if episode % 10 == 0:
                print(f"Episode {episode}: Total Reward = {sum(rewards)}")

In [6]:
env = gym.make('ALE/SpaceInvaders-v5', render_mode='rgb_array')
network = A2CNetwork(4, env.action_space.n)
optimizer = optim.Adam(network.parameters(), lr=1e-3)
agent = A2CAgent(env, network, optimizer)

agent.train(num_episodes=1000)

Episode 0: Total Reward = 20.0
Episode 10: Total Reward = 180.0
Episode 20: Total Reward = 285.0
Episode 30: Total Reward = 285.0
Episode 40: Total Reward = 285.0
Episode 50: Total Reward = 285.0
Episode 60: Total Reward = 245.0
Episode 70: Total Reward = 285.0
Episode 80: Total Reward = 285.0
Episode 90: Total Reward = 285.0
Episode 100: Total Reward = 135.0
Episode 110: Total Reward = 260.0
Episode 120: Total Reward = 285.0
Episode 130: Total Reward = 285.0
Episode 140: Total Reward = 210.0
Episode 150: Total Reward = 285.0
Episode 160: Total Reward = 210.0
Episode 170: Total Reward = 285.0
Episode 180: Total Reward = 210.0
Episode 190: Total Reward = 210.0
Episode 200: Total Reward = 285.0
Episode 210: Total Reward = 535.0
Episode 220: Total Reward = 285.0
Episode 230: Total Reward = 345.0
Episode 240: Total Reward = 285.0
Episode 250: Total Reward = 240.0
Episode 260: Total Reward = 230.0
Episode 270: Total Reward = 240.0
Episode 280: Total Reward = 260.0
Episode 290: Total Reward 

In [7]:
def play_trained_agent(env, network, episodes=5):
    for episode in range(episodes):
        state, _ = env.reset()
        state = preprocess_frame(state)
        stacked_frames = deque([np.zeros((84, 84), dtype=np.float32) for _ in range(4)], maxlen=4)
        stacked_state, stacked_frames = stack_frames(stacked_frames, state, is_new_episode=True)

        done = False
        total_reward = 0

        while not done:
            env.render()
            state_tensor = torch.FloatTensor(stacked_state).unsqueeze(0)
            policy, value = network(state_tensor)
            # print(policy.shape)
            action = torch.argmax(policy, dim=0).item()

            next_state, reward, done, _, _ = env.step(action)
            next_state = preprocess_frame(next_state)
            stacked_state, stacked_frames = stack_frames(stacked_frames, state, is_new_episode=False)

            total_reward += reward

        print(f"Episode {episode + 1} ended with total reward: {total_reward}")

In [8]:
env = gym.make("ALE/SpaceInvaders-v5", render_mode="human")
play_trained_agent(env, network)

Episode 1 ended with total reward: 270.0
Episode 2 ended with total reward: 270.0
Episode 3 ended with total reward: 270.0
Episode 4 ended with total reward: 270.0
Episode 5 ended with total reward: 270.0


In [9]:
s = [1., 2., 3.]
s = torch.tensor(s)
print(s)
s.sum()

tensor([1., 2., 3.])


tensor(6.)

In [10]:
a = [1, 2, 3]
b = ['a', 'b', 'c']
for i, j in zip(a, b):
    print(i, j)

1 a
2 b
3 c


In [11]:
returns = [1, 2]
returns.insert(0, 4)
returns

[4, 1, 2]

In [12]:
policy = torch.tensor([1., 6., 8., 18.])
probs = torch.softmax(policy, dim=-1)
m = Categorical(probs)
print(probs)
action = m.sample()
m.log_prob(action)

tensor([4.1397e-08, 6.1439e-06, 4.5398e-05, 9.9995e-01])


tensor(-5.1559e-05)

In [13]:
a = np.random.rand(4, 84, 84)
a = torch.from_numpy(a)
a = a.unsqueeze(0)
a.shape

torch.Size([1, 4, 84, 84])

In [ ]:
a = np.random.rand(4, 84, 84)
a = np.float32(a)
a = torch.from_numpy(a)
net = A2CNetwork(4, 2)
b = net(a)
b.shape

AttributeError: 'tuple' object has no attribute 'shape'

: 

In [ ]:
b.view(64, -1).shape

torch.Size([64, 49])

In [ ]:
a = np.array([1, 2])
np.float32(a)

array([1., 2.], dtype=float32)

In [ ]:
np.stack([[1, 2], [2, 3]], axis=0)

array([[1, 2],
       [2, 3]])